# Atlas-100M: one world model for weather, air quality and traffic (ParadoxAI Lab)

Rules, opponents and predictions: `atlas/m100/PREREG.md`, committed **before** this notebook first ran.

**Run:** Runtime → Change runtime type → **A100** → Save → **Run all**, allow Google Drive (company account).
Everything is a resumable queue saved to Drive `PWM-100M/`: after any disconnect, open the notebook and **Run all**
again — finished jobs are skipped and training continues from its last checkpoint. Expect several sessions
(≈ 50 A100-hours in total). The forecast archive for the hybrid stage downloads under a free daily limit and may need
re-runs on later days; everything else does not wait for it.

In [ ]:
# 1. Drive and GPU
import os, subprocess
from google.colab import drive
drive.mount('/content/drive', timeout_ms=300000)
STORE = '/content/drive/MyDrive/PWM-100M'; W10 = '/content/drive/MyDrive/PWM-10M'
assert os.path.exists(f'{W10}/data/W5K_X.npy'), 'the WEATHER-5K copy (PWM-10M/data) is missing from Drive'
assert os.path.exists(f'{W10}/live/W5K_OBS.npy'), 'the NOAA live-period copy (PWM-10M/live) is missing from Drive'
os.makedirs(STORE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code (public repository) and packages
!rm -rf /content/pwm && git clone -q --depth 1 -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model.git /content/pwm
!cd /content/pwm && git log --oneline -1
!pip install -q timesfm chronos-forecasting kagglehub 2>&1 | tail -1
!pip install -q --no-deps granite-tsfm tirex-ts && pip install -q einops deprecated 2>&1 | tail -1

In [ ]:
# 3. Data: WEATHER-5K and the live period from Drive, all-India CPCB from the public Kaggle copy, traffic from the
#    published sources (and the specialists' official code)
P = 'python -u /content/pwm/atlas/m100/pipeline.py --store ' + STORE + ' --w10 ' + W10
!{P} --stage data 2>&1 | tail -20

In [ ]:
# 4. Train Atlas-100M (resumes from Drive). Check G0 in the log: the grown model must reproduce the 24.1M exactly.
!{P} --stage train --cap 600 2>&1 | grep --line-buffered -v "Warning\|warn("

In [ ]:
# 5. Weather: full 2023 test, head-to-head (2023 and live) with the ten saved opponents, paired tests
!{P} --stage weather 2>&1 | grep --line-buffered -v "Warning\|warn(" | tail -80

In [ ]:
# 6. Air quality: nine foundation models zero-shot, Atlas-India v1.0, baselines, paired tests (Tracks S and U)
!{P} --stage aq 2>&1 | grep --line-buffered -v "Warning\|warn(" | tail -60

In [ ]:
# 7. Traffic: foundation models (Track B), STID / Graph WaveNet / STAEformer and PWM-Dyn v0.6r (Track A), paired tests
!{P} --stage traffic 2>&1 | grep --line-buffered -v "Warning\|warn(" | tail -80

In [ ]:
# 8. Hybrid: the global systems' archived forecasts (fetch resumes daily), then training and the 2025 test
!{P} --stage hybrid 2>&1 | grep --line-buffered -v "Warning\|warn(" | tail -80

In [ ]:
# 9. Summary of everything finished so far
import json, glob
OUT = f'{STORE}/out'
for f in ['atlas100m_summary.json', 'atlas100m_aq_results.json', 'atlas100m_traffic_results.json', 'atlas100m_hybrid_results.json']:
    p = f'{OUT}/{f}'
    print(('FOUND  ' if os.path.exists(p) else 'pending ') + f)
s = f'{OUT}/atlas100m_summary.json'
if os.path.exists(s):
    r = json.load(open(s)); print('parameters', r['params']['total'], '| best step', r['best_step'], '| val', r['best_val'], '| minutes', r['minutes_total'])
for f in sorted(glob.glob(f'{W10}/small/h2h/paired_atlas100m_*m100*.json')):
    print('paired:', os.path.basename(f))